# 14 Bidding Behaviour And Target Sensitivity Report

## How to use this notebook

Set `RUN_DIR` in the next cell to a completed Phase E3 run folder. The notebook reads saved outputs only. If `target_mode` is absent, it defaults to `current_soft_target` so the notebook can also open older E1/E2 runs.

In [ ]:
from pathlib import Path

RUN_DIR = Path(r"scripts/Data/03_Hydrogen_Test_Case/runs/REPLACE_WITH_RUN_ID")
RUN_DIR

In [ ]:
from IPython.display import Image, Markdown, display
import json
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DEFAULT_TARGET_MODE = "current_soft_target"

def _load_csv(name: str) -> pd.DataFrame:
    path = RUN_DIR / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

def _load_parquet(name: str) -> pd.DataFrame:
    path = RUN_DIR / name
    return pd.read_parquet(path) if path.exists() else pd.DataFrame()

def _with_target_mode(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    copy = frame.copy()
    if "target_mode" not in copy.columns:
        copy["target_mode"] = DEFAULT_TARGET_MODE
    return copy

def _show_images(pattern: str, limit: int | None = None) -> list[Path]:
    matches = sorted((RUN_DIR / "figures").glob(pattern))
    if limit is not None:
        matches = matches[:limit]
    for path in matches:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path)))
    return matches

daily_metrics = _with_target_mode(_load_csv("daily_metrics.csv"))
weekly_metrics = _with_target_mode(_load_csv("weekly_metrics.csv"))
weekly_metrics_by_mode = _with_target_mode(_load_csv("weekly_metrics_by_model_gamma_target_mode.csv"))
benchmark_metrics = _with_target_mode(_load_csv("benchmark_metrics.csv"))
perfect_foresight_metrics = _with_target_mode(_load_csv("perfect_foresight_metrics.csv"))
runtime_diagnostics = _with_target_mode(_load_csv("runtime_diagnostics.csv"))
runtime_summary = _load_csv("runtime_summary.csv")
validation_checks = _load_csv("validation_checks_all_runs.csv")
cvar_validation_checks = _load_csv("cvar_validation_checks.csv")
submitted_bids = _with_target_mode(_load_parquet("submitted_bids.parquet"))
actual_clearing = _with_target_mode(_load_parquet("actual_clearing.parquet"))
actual_redispatch = _with_target_mode(_load_parquet("actual_redispatch_timeseries.parquet"))
summary_manifest = json.loads((RUN_DIR / "production_target_sensitivity_manifest.json").read_text(encoding="utf-8")) if (RUN_DIR / "production_target_sensitivity_manifest.json").exists() else {}
selected_week_manifest = json.loads((RUN_DIR / "selected_week_manifest.json").read_text(encoding="utf-8")) if (RUN_DIR / "selected_week_manifest.json").exists() else {}
print("Loaded run:", RUN_DIR)

## Run scope and target modes

In [ ]:
scope = {
    "week": selected_week_manifest.get("selected_week", {}).get("week_label", "unknown"),
    "start": selected_week_manifest.get("selected_week", {}).get("delivery_start_date", "unknown"),
    "end": selected_week_manifest.get("selected_week", {}).get("delivery_end_date", "unknown"),
    "artifacts": sorted(daily_metrics.get("artifact_id", pd.Series(dtype=object)).astype(str).dropna().unique().tolist()),
    "models": sorted(daily_metrics.get("model_label", pd.Series(dtype=object)).astype(str).dropna().unique().tolist()),
    "gammas": sorted(pd.to_numeric(daily_metrics.get("cvar_gamma", pd.Series(dtype=float)), errors="coerce").dropna().unique().tolist()),
    "target_modes": sorted(daily_metrics.get("target_mode", pd.Series(dtype=object)).astype(str).dropna().unique().tolist()),
}
pd.Series(scope)

In [ ]:
pd.DataFrame(summary_manifest.get("target_modes", []))

## Validation and infeasibility summary

In [ ]:
if not validation_checks.empty:
    display(validation_checks.groupby(["status"], as_index=False).size())
    display(validation_checks.loc[validation_checks["status"].astype(str).isin(["fail", "warn"]), ["check_name", "status", "severity", "details"]])
if not cvar_validation_checks.empty:
    display(cvar_validation_checks.groupby(["target_mode", "status"], as_index=False).size())

if not daily_metrics.empty and "hard_target_infeasible" in daily_metrics.columns:
    display(
        daily_metrics.loc[daily_metrics["hard_target_infeasible"].astype(bool), [
            "model_label", "artifact_id", "delivery_day", "cvar_gamma", "target_mode",
            "stochastic_solver_status", "actual_redispatch_solver_status", "shortfall_kg"
        ]]
    )

## Headline target-sensitivity table

In [ ]:
headline_cols = [
    "model_label", "artifact_id", "cvar_gamma", "target_mode",
    "realised_adjusted_profit", "expected_adjusted_profit", "shortfall_kg",
    "production_fulfilment_ratio", "cvar_tail_profit", "worst_scenario_profit",
    "rejected_energy_mwh", "unused_cleared_energy_mwh",
    "weighted_average_bid_price", "high_bid_share", "market_cap_bid_share",
    "value_captured_vs_perfect_foresight", "solver_status"
]
headline = weekly_metrics_by_mode[headline_cols].sort_values(["model_label", "cvar_gamma", "target_mode"]) if not weekly_metrics_by_mode.empty else pd.DataFrame(columns=headline_cols)
headline

## Profit vs shortfall vs CVaR trade-off

In [ ]:
if not weekly_metrics_by_mode.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for model_label, group in weekly_metrics_by_mode.groupby("model_label", sort=False):
        axes[0].plot(group["cvar_gamma"], group["realised_adjusted_profit"], marker="o", label=model_label)
        axes[1].plot(group["cvar_gamma"], group["shortfall_kg"], marker="o", label=model_label)
        axes[2].plot(group["cvar_gamma"], group["cvar_tail_profit"], marker="o", label=model_label)
    axes[0].set_title("Realised profit")
    axes[1].set_title("Shortfall")
    axes[2].set_title("CVaR tail profit")
    for ax in axes:
        ax.set_xlabel("Gamma")
        ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

## Bid ladder and clearing diagnostics

In [ ]:
bid_figs = _show_images("fig_bid_ladder_actual_price_*.png", limit=6)
heatmap_figs = _show_images("fig_bid_ladder_heatmap_*.png", limit=6)
len(bid_figs), len(heatmap_figs)

## Cleared/used/unused electricity

In [ ]:
_show_images("fig_submitted_cleared_rejected_*.png", limit=6)
_show_images("fig_cleared_used_unused_*.png", limit=6)

## Asset operation examples

In [ ]:
_show_images("fig_asset_operation_*.png", limit=6)
_show_images("fig_target_mode_comparison_*.png", limit=6)

## Runtime diagnostics

In [ ]:
runtime_summary if not runtime_summary.empty else runtime_diagnostics.head()

In [ ]:
_show_images("fig_runtime_by_target_mode.png")
_show_images("fig_runtime_by_model_gamma.png")
_show_images("fig_runtime_vs_binary_count.png")
_show_images("fig_runtime_stage_breakdown.png")
_show_images("fig_top_slowest_solves.png")

## Interpretation and modelling recommendation

In [ ]:
if weekly_metrics_by_mode.empty:
    print("No weekly metrics available.")
else:
    ranking = weekly_metrics_by_mode.sort_values(["realised_adjusted_profit", "cvar_tail_profit"], ascending=[False, False])[
        ["model_label", "cvar_gamma", "target_mode", "realised_adjusted_profit", "cvar_tail_profit", "shortfall_kg", "solver_status"]
    ]
    display(ranking)
    best_profit = ranking.iloc[0]
    print(
        f"Highest realised profit: {best_profit['model_label']} gamma={best_profit['cvar_gamma']} target_mode={best_profit['target_mode']}"
    )
    if "hard_daily_target" in set(weekly_metrics_by_mode["target_mode"].astype(str)):
        hard_rows = weekly_metrics_by_mode.loc[weekly_metrics_by_mode["target_mode"].astype(str).eq("hard_daily_target")]
        display(hard_rows[["model_label", "cvar_gamma", "realised_adjusted_profit", "shortfall_kg", "solver_status"]])